In [119]:
# import libraries for sentiment analysis
import pandas as pd # read dataset 
import numpy as np # numeric operations
from textblob import TextBlob # get subjectivity for each text
import re # text cleaning
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # get VADER scores for each text
from sklearn.model_selection import train_test_split # split data into train and test sets
from sklearn.metrics import accuracy_score, classification_report # evaluate model performance
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # classifier to predict sentiment labels

from newsapi import NewsApiClient
import pandas as pd
import yfinance as yf
from pathlib import Path



In [120]:
headlines_path = Path("training_data/djia_news.csv")
djia_path = Path("training_data/djia_price_table.csv")


df1 = pd.read_csv(headlines_path, encoding="cp1252", low_memory=False, parse_dates=["Date"]) 
df2 = pd.read_csv(djia_path, encoding="cp1252", low_memory=False, parse_dates=["Date"]) 

In [121]:
print("df1:", df1.shape)
display(df1.head(3))


df1: (1989, 27)


,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as countries move to brink of war""",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into South Ossetia; footage from fighting (YouTube)',"b'Russian tanks are moving towards the capital of South Ossetia, which has reportedly been completely destroyed by Georgian artillery fire'","b""Afghan children raped with 'impunity,' U.N. official says - this is sick, a three year old was raped and they do nothing""",b'150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.',"b""Breaking: Georgia invades South Ossetia, Russia warned it would intervene on SO's side""","b""The 'enemy combatent' trials are nothing but a sham: Salim Haman has been sentenced to 5 1/2 years, but will be kept longer anyway just because they feel like it.""",...,"b'Georgia Invades South Ossetia - if Russia gets involved, will NATO absorb Georgia and unleash a full scale war?'",b'Al-Qaeda Faces Islamist Backlash',"b'Condoleezza Rice: ""The US would not act to prevent an Israeli strike on Iran."" Israeli Defense Minister Ehud Barak: ""Israel is prepared for uncompromising victory in the case of military hostili...",b'This is a busy day: The European Union has approved new sanctions against Iran in protest at its nuclear programme.',"b""Georgia will withdraw 1,000 soldiers from Iraq to help fight off Russian forces in Georgia's breakaway region of South Ossetia""",b'Why the Pentagon Thinks Attacking Iran is a Bad Idea - US News &amp; World Report',b'Caucasus in crisis: Georgia invades South Ossetia',"b'Indian shoe manufactory - And again in a series of ""you do not like your work?""'",b'Visitors Suffering from Mental Illnesses Banned from Olympics',"b""No Help for Mexico's Kidnapping Surge"""
1,2008-08-11,1,"b'Why wont America and Nato help us? If they wont help us now, why did we help them in Iraq?'",b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli training, we're fending off Russia """,b'Georgian army flees in disarray as Russians advance - Gori abandoned to Russia without a shot fired',"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zealand Passports doing in Iraq?',b'Russia angered by Israeli military sale to Georgia',b'An American citizen living in S.Ossetia blames U.S. and Georgian leaders for the genocide of innocent people',...,b'Israel and the US behind the Georgian aggression?',"b'""Do not believe TV, neither Russian nor Georgian. There are much more victims""'",b'Riots are still going on in Montreal (Canada) because police murdered a boy on Saturday.',b'China to overtake US as largest manufacturer',b'War in South Ossetia [PICS]',b'Israeli Physicians Group Condemns State Torture',b' Russia has just beaten the United States over the head with Peak Oil',b'Perhaps *the* question about the Georgia - Russia conflict ',b'Russia is so much better at war',"b""So this is what it's come to: trading sex for food."""
2,2008-08-12,0,"b'Remember that adorable 9-year-old who sang at the opening ceremonies? That was fake, too.'","b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would have no children...""'","b""Al-Qa'eda is losing support in Iraq because of a brutal crackdown on activities it regards as un-Islamic - including women buying cucumbers""",b'Ceasefire in Georgia: Putin Outmaneuvers the West',b'Why Microsoft and Intel tried to kill the XO $100 laptop',b'Stratfor: The Russo-Georgian War and the Balance of Power ',"b""I'm Trying to Get a Sense of This Whole Georgia-Russia War: Vote Up If You Think Georgia Started It, Or Down If you Think Russia Did""",...,b'U.S. troops still in Georgia (did you know they were in Georgia in the first place?)',b'Why Russias response to Georgia was right',"b'Gorbachev accuses U.S. of ma

In [122]:
print("df2:", df2.shape)
display(df2.head(3))


df2: (1989, 7)


,Date,Open,High,Low,Close,Volume,Adj Close
0,2016-07-01,17924.240234,18002.380859,17916.910156,17949.369141,82160000,17949.369141
1,2016-06-30,17712.759766,17930.609375,17711.800781,17929.990234,133030000,17929.990234
2,2016-06-29,17456.019531,17704.509766,17456.019531,17694.679688,106380000,17694.679688


In [ ]:
# create a new dataset merging headlines and DJIA data on Date
merge = df1.merge(df2, how="inner", on="Date")

# show the mergede dataset
merge

In [125]:
# combine news headlines into one column 
headlines = []

# for each row in the dataframe, combine the news headlines into one string - iloc is used to access the rows and columns by index 
for row in range(0,len(merge.index)):
    headlines.append(' '.join(str(x) for x in merge.iloc[row, 2:27])) # (25 news headlines from column 2 to 26)


In [ ]:
# print a sample of the combined headlines 
headlines[0]

'b"Georgia \'downs two Russian warplanes\' as countries move to brink of war" b\'BREAKING: Musharraf to be impeached.\' b\'Russia Today: Columns of troops roll into South Ossetia; footage from fighting (YouTube)\' b\'Russian tanks are moving towards the capital of South Ossetia, which has reportedly been completely destroyed by Georgian artillery fire\' b"Afghan children raped with \'impunity,\' U.N. official says - this is sick, a three year old was raped and they do nothing" b\'150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.\' b"Breaking: Georgia invades South Ossetia, Russia warned it would intervene on SO\'s side" b"The \'enemy combatent\' trials are nothing but a sham: Salim Haman has been sentenced to 5 1/2 years, but will be kept longer anyway just because they feel like it." b\'Georgian troops retreat from S. Osettain capital, presumably leaving several hundred people killed. [VIDEO]\' b\'Did the U.S. Prep Georgia for War with Russia?\'

In [129]:
# clean the dataset 
clean_headlines = []

for i in range(0,len(headlines)):
    # replace non-alphabetic characters with spaces
    clean_headlines.append(re.sub("b[(')]", ' ', headlines[i])) # remove b'
    clean_headlines[i] = re.sub('b[(")]', ' ', clean_headlines[i]) # remove b"
    clean_headlines[i] = re.sub("[\ ']", ' ', clean_headlines[i]) # remove \'

<>:8: SyntaxWarning: invalid escape sequence '\ '
<>:8: SyntaxWarning: invalid escape sequence '\ '
C:\Users\35387\AppData\Local\Temp\ipykernel_18072\4110210003.py:8: SyntaxWarning: invalid escape sequence '\ '
  clean_headlines[i] = re.sub("[\ ']", ' ', clean_headlines[i]) # remove \'


In [ ]:
# show the combined cleaned headlines 
clean_headlines[20]

' A French judge has ordered two branches of Scientologists and their leaders to stand trial for fraud     Russia in legal bid to ban South Park    60 Minutes  Cut Ahmadinejad s Statement,  Solution Is Democracy  in Israel/Palestine"  U.S. drones kill 13 in missile attack in Pakistan   Screw You, TSA: No Conviction on Key Charges in Liquid-Bomb Trial in London   Scientology on trial for fraud in France!   An EU ban on ads with sexist overtones? Another quasi-fictional piece of translucent flimflam   Film Backs Afghans Claims of US Killings [of 90+ civilians]   Giant Buddha found at Afghan site.   After denying strenously the US reopens inquiry into Afghan attack that may have killed upto 90 civilians   Videos surface showing dead Afghan children after US raid, sparking a new investigation   "Consortium" of Media Execs to Canadian Green Party:  You can\\ t participate in debate because the other parties don\\ t want you there.   Everything going wrong in the world .. in one convenient g

In [132]:
# add clean headlines to the merge dataset 
merge['Combined News'] = clean_headlines

merge['Combined News'][0]

' Georgia  downs two Russian warplanes  as countries move to brink of war"  BREAKING: Musharraf to be impeached.   Russia Today: Columns of troops roll into South Ossetia; footage from fighting (YouTube)   Russian tanks are moving towards the capital of South Ossetia, which has reportedly been completely destroyed by Georgian artillery fire   Afghan children raped with  impunity,  U.N. official says - this is sick, a three year old was raped and they do nothing"  150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.   Breaking: Georgia invades South Ossetia, Russia warned it would intervene on SO s side"  The  enemy combatent  trials are nothing but a sham: Salim Haman has been sentenced to 5 1/2 years, but will be kept longer anyway just because they feel like it."  Georgian troops retreat from S. Osettain capital, presumably leaving several hundred people killed. [VIDEO]   Did the U.S. Prep Georgia for War with Russia?   Rice Gives Green Light for 